# 01 - Attention Heads

This notebook builds beginner intuition for attention heads in transformers.

Audience: a software engineer learning mechanistic interpretability.

The goal is clarity, not completeness. We will use toy examples and diagrams, avoid heavy math, and focus on how attention can help us ask better questions about model behavior.

## Mental Model

A transformer processes text as a sequence of tokens. At each layer, every token has a current internal representation. Attention heads help each token decide which other token representations to read from.

A useful beginner mental model:

```text
token position receiving information
        |
        v
  attention head asks: which previous/current tokens should this token read from?
        |
        v
mixed information is written back into the token representation
```

This is not the whole transformer. It is one routing step inside a larger computation.

## What is an attention head?

An attention head is a small information-routing component inside a transformer layer.

For each token position, the head decides how much information to read from other token positions. It then mixes information from those positions into the current token's representation.

A software analogy: an attention head is less like a function that returns the final answer and more like a routing rule that decides which records in memory are relevant to the current record.

```text
Sentence:  The cat sat because it was tired

Question for token "it":
Which earlier token might explain what "it" refers to?

Possible attention route:
it  ---------------------->  cat
```

The head does not literally say "it means cat" in English. It moves information in a way that may help later parts of the model represent that relationship.

### What I would have believed before learning this

I would have believed an attention head directly understands or explains a sentence. A better view is that a head routes information between token positions. Understanding may emerge from many such routing and transformation steps working together.

## Attention head inputs and outputs

For every token, an attention head works with three beginner-friendly roles:

| Role | Intuition | Plain-English question |
| --- | --- | --- |
| Query | What this token is looking for | What kind of information do I need? |
| Key | What each token can be matched by | Do I look relevant to that query? |
| Value | What information gets moved | If selected, what information do I contribute? |

You do not need the matrix math yet. The important idea is that each token compares its query against other tokens' keys, then uses the resulting attention pattern to mix values.

```text
current token query
        |
        v
compare against keys from other tokens
        |
        v
attention pattern: mostly read from these positions
        |
        v
mix their values into the current token
```

### What I would have believed before learning this

I would have believed attention was just a weighted word lookup. That is close enough to start, but incomplete: the lookup happens over learned internal representations, not raw words, and the information being moved may not be human-readable.

## Why transformers use multiple heads

A single attention head can only express one attention pattern at a time for each layer. Multiple heads give the model several parallel ways to route information.

Different heads might specialize in different rough patterns:

- looking at the previous token
- looking at punctuation or separators
- connecting a pronoun to a noun
- tracking repeated words
- gathering local context from nearby tokens

These descriptions are simplifications. A real head may not have a clean name. But the core idea is useful: multiple heads increase the model's ability to move different kinds of information at the same layer.

```text
Layer N

Head A: route local context
Head B: route delimiter information
Head C: route repeated-token information
Head D: route subject-related information

All outputs are combined back into the token representation.
```

### What I would have believed before learning this

I would have believed more heads simply means the model pays attention to more words. A better view is that multiple heads give the model multiple routing channels, each of which can carry a different kind of signal.

## What attention patterns can reveal

An attention pattern shows how strongly each destination token attends to each source token.

For interpretability, this can reveal useful clues:

- whether a token reads from nearby context or distant context
- whether a head tracks repeated tokens or separators
- whether a head often attends to syntactic anchors like names, punctuation, or previous words
- whether behavior changes across different prompts

A tiny example attention table:

| Destination token | Strongest source token | Possible interpretation |
| --- | --- | --- |
| `it` | `cat` | The head may be routing noun information into the pronoun position. |
| `tired` | `was` | The head may be routing local grammar context. |
| `.` | `tired` | The head may be reading the end of the statement. |

The phrase "may be" matters. Attention patterns are evidence, not proof.

### What I would have believed before learning this

I would have believed a bright cell in an attention heatmap directly tells me why the model answered the way it did. A better habit is to treat a bright cell as a lead to investigate, not the final explanation.

## What attention patterns do NOT explain

Attention patterns are useful, but they are not complete explanations.

They do not directly tell us:

- what exact information is stored in each token representation
- whether the attended information was necessary for the final output
- how later layers transformed or ignored the routed information
- why the model chose one final token instead of another
- whether the pattern is stable across similar examples

A common mistake is to look at an attention map and say, "The model attended to token X, therefore token X caused the answer." That may be true, false, or only partly true. The attention pattern only shows one information route inside a larger system.

### What I would have believed before learning this

I would have believed attention maps were explanations. Now I would call them diagnostic views: they show where information might flow, but not the complete computation or causal story.

## Why attention is useful for interpretability but incomplete

Attention is useful because it gives us a visible handle on information movement. If a head repeatedly routes from pronouns to nouns, separators to section starts, or repeated tokens to earlier copies, that pattern can help us form a concrete hypothesis about what the model is doing.

But attention is incomplete because the transformer is not only routing information. It is also transforming representations across many layers. The model's final behavior depends on how routed information is combined, changed, suppressed, or amplified later.

A practical rule:

> Use attention patterns to generate hypotheses. Do not use them alone as final explanations.

### What I would have believed before learning this

I would have believed interpretability starts by finding the right heatmap. A better view is that a heatmap is one instrument on the dashboard: useful, but not the whole engine.

## How this connects to mechanistic interpretability

Mechanistic interpretability tries to understand model behavior by studying the internal mechanisms that produce it.

Researchers study attention heads because they are relatively inspectable components. A head has a pattern: for each token, it attends to other token positions. That makes it easier to notice repeated behaviors, compare examples, and ask whether a head seems to route a specific kind of information.

Attention can reveal:

- which token positions are connected by a head
- whether a head behaves similarly across examples
- whether a head appears to track structure such as repetition, locality, punctuation, or references
- where to look next when debugging model behavior

Attention cannot reveal the full mechanism by itself. It does not tell us the full meaning of the representations being moved, how later layers use them, or whether a pattern is causally responsible for the output.

For a beginner, the most important habit is humility: attention patterns are clues. Good interpretability work keeps asking what the clue does and does not justify.

### What I would have believed before learning this

I would have believed mechanistic interpretability meant naming heads and declaring what they do. A better beginner goal is to form careful hypotheses from patterns, then avoid claiming more than the evidence supports.

## Simple worked example using a toy sentence

Toy sentence:

```text
The cat sat because it was tired.
```

Token list, simplified:

| Position | Token |
| ---: | --- |
| 0 | The |
| 1 | cat |
| 2 | sat |
| 3 | because |
| 4 | it |
| 5 | was |
| 6 | tired |
| 7 | . |

Imagine one attention head has the following strongest routes:

| Destination token | Reads most from | Beginner hypothesis |
| --- | --- | --- |
| `cat` | `The` | Local noun phrase context. |
| `sat` | `cat` | Subject information may be routed into the verb. |
| `it` | `cat` | Pronoun position may receive noun information. |
| `tired` | `it` | Predicate may read from the pronoun. |
| `.` | `tired` | End punctuation may read from the final content word. |

A rough diagram:

```text
The   cat   sat   because   it   was   tired   .
 |     ^      ^              ^            ^     ^
 |_____|      |______________|            |_____|
```

How to read this carefully:

- The pattern suggests possible information routes.
- It does not prove the model resolved the pronoun correctly.
- It does not show what exact features are stored at `cat` or `it`.
- It does not show how later layers use this routed information.

A good beginner interpretation is modest: this hypothetical head appears to route some noun-related context toward later tokens that might need it.

### What I would have believed before learning this

I would have believed the route from `it` to `cat` proves the model knows the pronoun refers to the cat. A better interpretation is narrower: the pattern is consistent with that possibility, but it is not enough evidence by itself.

## Recap

- Attention heads route information between token positions.
- Queries, keys, and values are a useful mental model for matching and moving information.
- Multiple heads give the model multiple routing channels.
- Attention patterns can reveal useful clues about information flow.
- Attention patterns do not fully explain model behavior.
- Mechanistic interpretability requires careful claims: attention is evidence, not a complete explanation.

## Next questions

- Which simple attention patterns appear often across many sentences?
- How do attention outputs become part of the residual stream?
- When does a head have a stable role versus a context-dependent role?
- What additional evidence would make an attention-based hypothesis stronger?